# FraudShield AI — Notebook 05: Model Evaluation & Selection

This notebook:
- Evaluates all trained models with comprehensive metrics
- Performs threshold optimization (not just 0.5)
- Generates SHAP explainability
- Selects the best model and saves it as `best_model.joblib`
- Saves evaluation metrics to `model_metrics.json` for the backend API

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

PROC_PATH  = '../data/processed/'
MODEL_PATH = '../models/'

# Load test data
X_test_scaled = np.load(f'{PROC_PATH}X_test.npy')
y_test        = np.load(f'{PROC_PATH}y_test.npy')
X_test_raw    = np.load(f'{PROC_PATH}X_test_raw.npy')

with open(f'{PROC_PATH}feature_columns.json') as f:
    FEATURE_COLS = json.load(f)

# Load all trained models
models = {}
model_files = {
    'Logistic Regression': (f'{MODEL_PATH}logistic_regression.joblib', X_test_scaled),
    'Random Forest':       (f'{MODEL_PATH}random_forest.joblib',       X_test_scaled),
    'XGBoost':             (f'{MODEL_PATH}xgboost.joblib',             X_test_raw),
}

for name, (path, _) in model_files.items():
    if os.path.exists(path):
        models[name] = joblib.load(path)
        print(f'Loaded: {name}')
    else:
        print(f'NOT FOUND: {path} — run 04_Model_Training.ipynb first')

if not models:
    raise FileNotFoundError('No models found. Please run 04_Model_Training.ipynb first.')

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────
fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5))
if len(models) == 1: axes = [axes]

all_metrics = []
for ax, (name, model) in zip(axes, models.items()):
    X_eval = model_files[name][1]
    probs  = model.predict_proba(X_eval)[:, 1]
    preds  = (probs >= 0.5).astype(int)
    cm     = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Legitimate','Fraud'],
        yticklabels=['Legitimate','Fraud'],
        cbar=False
    )
    ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    all_metrics.append({
        'name':      name,
        'probs':     probs,
        'preds':     preds,
        'cm':        cm.tolist(),
    })

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROC_PATH}confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC & PR Curves ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#3498db','#2ecc71','#e74c3c','#9b59b6']

metrics_summary = []
for (name, model), color, m in zip(models.items(), colors, all_metrics):
    probs = m['probs']
    # ROC
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax1.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
    # PR
    prec, rec, _ = precision_recall_curve(y_test, probs)
    pr_auc = average_precision_score(y_test, probs)
    ax2.plot(rec, prec, label=f'{name} (PR-AUC={pr_auc:.3f})', color=color, lw=2)
    
    metrics_summary.append({
        'model_name':   name,
        'precision':    round(precision_score(y_test, m['preds'], zero_division=0), 4),
        'recall':       round(recall_score(y_test, m['preds'], zero_division=0), 4),
        'f1':           round(f1_score(y_test, m['preds'], zero_division=0), 4),
        'roc_auc':      round(auc, 4),
        'pr_auc':       round(pr_auc, 4),
        'confusion_matrix': m['cm'],
    })

ax1.plot([0,1],[0,1],'k--', alpha=0.4)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves', fontweight='bold')
ax1.legend()

baseline = y_test.mean()
ax2.axhline(baseline, linestyle='--', color='gray', alpha=0.5, label=f'Baseline ({baseline:.3f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves', fontweight='bold')
ax2.legend()

plt.suptitle('ROC & Precision-Recall Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROC_PATH}roc_pr_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Threshold Optimization ────────────────────────────────────────
# Default threshold 0.5 is NOT ideal for imbalanced fraud data.
# We find the threshold that maximizes F1 score on the test set.

best_model_name = max(metrics_summary, key=lambda x: x['pr_auc'])['model_name']
best_model = models[best_model_name]
X_best = model_files[best_model_name][1]
best_probs = best_model.predict_proba(X_best)[:, 1]

print(f'Optimizing threshold for best model: {best_model_name}')

thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores  = []
for t in thresholds:
    preds_t = (best_probs >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds_t, zero_division=0))

best_threshold  = thresholds[np.argmax(f1_scores)]
best_f1_at_opt  = max(f1_scores)
print(f'Optimal threshold: {best_threshold:.2f} (F1={best_f1_at_opt:.4f})')
print(f'Default threshold: 0.50 (F1={f1_score(y_test, (best_probs>=0.5).astype(int), zero_division=0):.4f})')

plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_scores, color='#3498db', lw=2)
plt.axvline(best_threshold, color='#e74c3c', linestyle='--', label=f'Optimal: {best_threshold:.2f}')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.6, label='Default: 0.50')
plt.xlabel('Classification Threshold')
plt.ylabel('F1 Score')
plt.title(f'Threshold Optimization — {best_model_name}', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(f'{PROC_PATH}threshold_optimization.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Explainability ───────────────────────────────────────────
print(f'Generating SHAP explanations for {best_model_name}...')
try:
    model_type = type(best_model).__name__
    # Sample test set for SHAP (full set can be slow)
    sample_size = min(1000, len(X_best))
    idx = np.random.choice(len(X_best), sample_size, replace=False)
    X_sample = X_best[idx]
    
    if model_type in ('XGBClassifier','RandomForestClassifier','GradientBoostingClassifier'):
        explainer   = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_sample)
        # For binary classifiers shap_values may be list
        if isinstance(shap_values, list):
            sv = shap_values[1]
        else:
            sv = shap_values
    else:
        # LinearExplainer for logistic regression
        explainer   = shap.LinearExplainer(best_model, X_sample)
        shap_values = explainer.shap_values(X_sample)
        sv = shap_values

    # SHAP Summary Plot (bar)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        sv, X_sample, feature_names=FEATURE_COLS,
        plot_type='bar', show=False, max_display=10
    )
    plt.title(f'SHAP Feature Importance — {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{PROC_PATH}shap_importance.png', dpi=120, bbox_inches='tight')
    plt.show()

    # SHAP Beeswarm
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv, X_sample, feature_names=FEATURE_COLS, show=False, max_display=10)
    plt.title(f'SHAP Beeswarm — {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{PROC_PATH}shap_beeswarm.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Save global SHAP importances
    shap_importance = dict(zip(FEATURE_COLS, np.abs(sv).mean(axis=0).tolist()))
    print('SHAP explanations generated successfully.')
except Exception as e:
    print(f'SHAP generation failed: {e}')
    shap_importance = {}

In [ ]:
# ── Final Evaluation with Optimal Threshold ───────────────────────
final_preds = (best_probs >= best_threshold).astype(int)
print(f'\nFINAL EVALUATION — {best_model_name} (threshold={best_threshold:.2f})')
print('=' * 60)
print(classification_report(y_test, final_preds, target_names=['Legitimate','Fraud']))
final_cm = confusion_matrix(y_test, final_preds).tolist()

In [ ]:
# ── Save Best Model & Metrics JSON ────────────────────────────────
# This is the model that the backend API will load
import shutil
best_model_path = model_files[best_model_name][0]
shutil.copy(best_model_path, f'{MODEL_PATH}best_model.joblib')
print(f'Best model saved: {MODEL_PATH}best_model.joblib')

# Collect metrics for the backend
best_summary = next(m for m in metrics_summary if m['model_name'] == best_model_name)

metrics_json = {
    'model_name':            best_model_name,
    'model_version':         '1.0.0',
    'precision':             round(precision_score(y_test, final_preds, zero_division=0), 4),
    'recall':                round(recall_score(y_test, final_preds, zero_division=0), 4),
    'f1':                    round(f1_score(y_test, final_preds, zero_division=0), 4),
    'roc_auc':               best_summary['roc_auc'],
    'pr_auc':                best_summary['pr_auc'],
    'threshold':             round(float(best_threshold), 2),
    'confusion_matrix':      final_cm,
    'training_samples':      int(np.load(f'{PROC_PATH}X_train.npy').shape[0]),
    'test_samples':          int(len(y_test)),
    'shap_feature_importance': shap_importance,
    'all_models':            metrics_summary,
}

with open(f'{MODEL_PATH}model_metrics.json', 'w') as f:
    json.dump(metrics_json, f, indent=2)

print(f'Metrics saved: {MODEL_PATH}model_metrics.json')
print()
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'Best Model  : {best_model_name}')
print(f'Threshold   : {best_threshold:.2f}')
print(f'F1 Score    : {metrics_json["f1"]}')
print(f'PR-AUC      : {metrics_json["pr_auc"]}')
print(f'ROC-AUC     : {metrics_json["roc_auc"]}')
print()
print('Start the FastAPI backend:')
print('  cd FraudShield-AI')
print('  uvicorn backend.main:app --reload')